# RNN Location Extraction — Final Submission

**Verified Kaggle public score: 0.59388** (original postprocessed submission). This notebook retains the two trained RNNs and the winning inference settings. The score is historical; reruns require the same trained checkpoints and dataset.

Set `TRAIN_IF_MISSING=False` to reuse saved checkpoints. The final CSV is generated without retraining.

## 1. Setup

In [ ]:
import os, sys, subprocess, zipfile, shutil
from pathlib import Path

TRAIN_IF_MISSING = False
RUN_UK_AUDIT = False
USE_DRIVE_BACKUP = False

ROOT = Path('/content/rnn_competition')
DATA_DIR = ROOT / 'data'
OUT_DIR = ROOT / 'output'
MODEL_DIR = OUT_DIR / 'models'
for folder in (DATA_DIR, OUT_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lightning>=2.3,<3', 'kaggle>=1.7,<2'
], check=True)

import torch
from google.colab import userdata

if USE_DRIVE_BACKUP:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_models = Path('/content/drive/MyDrive/rnn_competition_output/models')
    for name in ('GRU_all', 'LSTM_all'):
        destination = MODEL_DIR / name
        destination.mkdir(parents=True, exist_ok=True)
        for filename in ('best.ckpt', 'encoder.pkl'):
            source = drive_models / name / filename
            if source.exists() and not (destination / filename).exists():
                shutil.copy2(source, destination / filename)

needed = ('uk_geo_dataset.csv', 'ru_geo_dataset.csv', 'labeling_sample.csv', 'test.csv')
needed_now = needed if TRAIN_IF_MISSING else ('uk_geo_dataset.csv', 'labeling_sample.csv', 'test.csv')
if not all((DATA_DIR / f).exists() for f in needed_now):
    token = userdata.get('KAGGLE_API_TOKEN')
    if not token:
        raise RuntimeError('Create KAGGLE_API_TOKEN in Colab Secrets and grant access')
    os.environ['KAGGLE_API_TOKEN'] = token
    os.environ.pop('KAGGLE_USERNAME', None)
    os.environ.pop('KAGGLE_KEY', None)
    result = subprocess.run(
        ['kaggle', 'competitions', 'download',
         '-c', 'kse-ua-location-extraction-2026', '-p', str(DATA_DIR)],
        capture_output=True, text=True
    )
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    for archive in DATA_DIR.glob('*.zip'):
        with zipfile.ZipFile(archive) as z:
            z.extractall(DATA_DIR)
        archive.unlink()

missing = [str(DATA_DIR / f) for f in needed_now if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Dataset files missing: {missing}')

GPU = torch.cuda.is_available()
DEVICE = 'cuda' if GPU else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if GPU else 'CPU')
print('Checkpoint directory:', MODEL_DIR)

## 2. Data and evaluation

In [ ]:
import re, ast, json, gc, pickle, string
from collections import Counter, defaultdict
from typing import List
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import lightning.pytorch as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger

SEED = 42
L.seed_everything(SEED, workers=True)
NUM_WORKERS = 2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.set_float32_matmul_precision('high')

In [ ]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p,  = 0.0, 0.0, 0.0, 0.0

    for y_true_sample, y_pred_sample in zip(y_true, y_pred):
        y_true_sample = set([hard_processing(s) for s in y_true_sample])
        y_pred_sample = set([hard_processing(s) for s in y_pred_sample])

        tp += len(y_true_sample & y_pred_sample)
        fp += len(y_pred_sample - y_true_sample)
        fn += len(y_true_sample - y_pred_sample)
        p += len(y_true_sample)

    if tp + fp == 0:
        if p == 0:
            precision = 1.0
        else:
            precision = 0.0
    else:
        precision = tp/(tp + fp)

    if tp + fn == 0:
        if p == 0:
            recall = 1.0
        else:
            recall = 0.0
    else:
        recall = tp/(tp + fn)

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return round(f1, 8)

def make_gt(df):
    return [[text[l:r] for l, r in spans]
            for text, spans in zip(df['text'], df['loc_markers'])]

In [ ]:
def to_list(value):
    return ast.literal_eval(value) if isinstance(value, str) else value

MARK_COLS = ['loc_markers', 'org_markers', 'per_markers']
KEEP = ['text'] + MARK_COLS

uk_df = pd.read_csv(DATA_DIR / 'uk_geo_dataset.csv')
labels_df = pd.read_csv(DATA_DIR / 'labeling_sample.csv')
TEST = pd.read_csv(DATA_DIR / 'test.csv').assign(text=lambda df: df.text.fillna('').astype(str))

for col in MARK_COLS:
    uk_df[col] = uk_df[col].apply(to_list)
labels_df['loc_markers'] = labels_df['loc_markers'].apply(to_list)
labels_df['locations'] = labels_df['locations'].apply(to_list)

uk_valid = uk_df[uk_df.is_valid == 1].drop_duplicates('text')[KEEP].reset_index(drop=True)
lab = labels_df[['text', 'loc_markers']].copy()
lab['org_markers'] = [[] for _ in range(len(lab))]
lab['per_markers'] = [[] for _ in range(len(lab))]
lab = lab[KEEP].reset_index(drop=True)

VAL_SETS = {'uk': uk_valid, 'lab': lab}
VAL_GOLD = {k: make_gt(df) for k, df in VAL_SETS.items()}
EVAL_TEXTS = {'uk': uk_valid.text.tolist(), 'lab': lab.text.tolist(), 'test': TEST.text.tolist()}

if TRAIN_IF_MISSING:
    ru_df = pd.read_csv(DATA_DIR / 'ru_geo_dataset.csv')
    for col in MARK_COLS:
        ru_df[col] = ru_df[col].apply(to_list)
    uk_tr = uk_df[uk_df.is_valid == 0].drop_duplicates('text')[KEEP].reset_index(drop=True)
    ru_tr = ru_df.drop_duplicates('text')[KEEP].reset_index(drop=True)

    def take(df, n):
        return df if n is None or n >= len(df) else df.sample(n, random_state=SEED)

    def train_frame(cfg):
        pieces = [take(uk_tr, cfg['n_uk'])]
        if cfg['n_ru']:
            pieces.append(take(ru_tr, cfg['n_ru']))
        return pd.concat(pieces, ignore_index=True)

print('UK validation:', len(uk_valid), 'labeled:', len(lab), 'test:', len(TEST))

## 3. Tokenizer and labels

In [ ]:
WORD_RE = re.compile(r"\w+(?:['’ʼ`]\w+)*|[^\w\s]")
PAD, UNK = 0, 1

def word_spans(text, pattern=WORD_RE):
    return [(m.start(), m.end()) for m in pattern.finditer(text)]

class WordTokenizer:
    kind = 'word'

    def __init__(self, min_freq=2, max_size=200_000):
        self.min_freq, self.max_size = min_freq, max_size

    def fit(self, texts):
        cnt = Counter(w.lower() for t in texts for w in WORD_RE.findall(t))
        self.itos = ['<pad>', '<unk>'] + [w for w, c in cnt.most_common(self.max_size) if c >= self.min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        return self

    @property
    def vocab_size(self):
        return len(self.itos)

    def encode(self, text):

        ws = word_spans(text)
        return [self.stoi.get(text[l:r].lower(), UNK) for l, r in ws], ws, list(range(len(ws))), ws

In [ ]:
TAG_SETS = {'all': ['LOC', 'ORG', 'PER'], 'loc': ['LOC']}
N_CASE = 7

def case_id(tok):
    if not any(c.isalpha() for c in tok):
        return 5 if any(c.isdigit() for c in tok) else 6
    if tok.isupper() and len(tok) > 1:
        return 3
    if tok.istitle():
        return 2
    return 1 if tok.islower() else 4

def spans_to_bio(offsets, spans, etypes, tag_id):

    labels = [0] * len(offsets)
    for e in etypes:
        for l, r in spans.get(e, []):
            inside = False
            for i, (a, b) in enumerate(offsets):
                if a >= r:
                    break
                if b > l and labels[i] == 0:
                    labels[i] = tag_id[('I-' if inside else 'B-') + e]
                    inside = True
    return labels

class Encoder:

    def __init__(self, tokenizer, etypes, train_texts, max_word_len=20, min_char_freq=5):
        self.tok, self.etypes, self.max_word_len = tokenizer, etypes, max_word_len
        self.tags = ['O'] + [f'{p}-{e}' for e in etypes for p in 'BI']
        assert self.tags[1:3] == ['B-LOC', 'I-LOC']
        self.tag_id = {t: i for i, t in enumerate(self.tags)}
        cnt = Counter(c for t in train_texts[:200_000] for c in t)
        self.char_itos = ['<pad>', '<unk>'] + [c for c, n in cnt.most_common(1000) if n >= min_char_freq]
        self.char_stoi = {c: i for i, c in enumerate(self.char_itos)}

    def __call__(self, text, spans=None, max_len=None):
        ids, offs, wids, ws = self.tok.encode(text)
        if not ids:
            ids, offs, wids, ws = [UNK], [(0, 0)], [0], [(0, 0)]
        if max_len and len(ids) > max_len:
            ids, offs, wids = ids[:max_len], offs[:max_len], wids[:max_len]
            ws = ws[:wids[-1] + 1]
        chars = [[self.char_stoi.get(c, UNK) for c in text[l:r][:self.max_word_len]] or [UNK] for l, r in offs]
        case = [case_id(text[l:r]) for l, r in offs]
        labels = None if spans is None else spans_to_bio(offs, spans, self.etypes, self.tag_id)
        return dict(ids=ids, chars=chars, case=case, labels=labels, word_ids=wids, word_spans=ws, text=text)

## 4. Dataset and batching

In [ ]:
def pack(strings):

    b = [s.encode('utf-8') for s in strings]
    offs = np.zeros(len(b) + 1, np.int64)
    offs[1:] = np.cumsum([len(x) for x in b])
    return np.frombuffer(b''.join(b), np.uint8), offs

def unpack(packed, i):
    buf, offs = packed
    return buf[offs[i]:offs[i + 1]].tobytes().decode('utf-8')

class NERDataset(Dataset):

    def __init__(self, df, encoder, max_len=None, labelled=True):
        self.enc, self.max_len, self.n = encoder, max_len, len(df)
        self._texts = pack(df['text'].fillna('').astype(str).tolist())
        self._spans = pack([json.dumps([list(a), list(b), list(c)]) for a, b, c in
                            zip(df['loc_markers'], df['org_markers'], df['per_markers'])]) if labelled else None

    @property
    def texts(self):
        return [unpack(self._texts, i) for i in range(self.n)]

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        spans = None if self._spans is None else dict(zip(('LOC', 'ORG', 'PER'), json.loads(unpack(self._spans, i))))
        return {**self.enc(unpack(self._texts, i), spans, self.max_len), 'idx': i}

In [ ]:
def collate(batch):
    lens = [len(b['ids']) for b in batch]
    B, T = len(batch), max(lens)
    W = max(len(c) for b in batch for c in b['chars'])
    ids, case = np.zeros((B, T), np.int64), np.zeros((B, T), np.int64)
    chars, labels = np.zeros((B, T, W), np.int64), np.full((B, T), -100, np.int64)
    for i, b in enumerate(batch):
        n = lens[i]
        ids[i, :n], case[i, :n] = b['ids'], b['case']
        if b['labels'] is not None:
            labels[i, :n] = b['labels']
        for j, c in enumerate(b['chars']):
            chars[i, j, :len(c)] = c
    tensors = dict(ids=ids, chars=chars, case=case, labels=labels)
    meta = {k: [b[k] for b in batch] for k in ('idx', 'word_ids', 'word_spans', 'text')}
    return {**{k: torch.from_numpy(v) for k, v in tensors.items()}, 'lengths': torch.tensor(lens), **meta}

def make_loader(ds, batch_size, shuffle):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate, num_workers=NUM_WORKERS,
                      persistent_workers=shuffle and NUM_WORKERS > 0,
                      pin_memory=GPU)

## 5. Decoding and RNN

In [ ]:
def word_level_probs(p, word_ids, n_words):

    wids = np.asarray(word_ids)
    loc = p[:, 1] + p[:, 2]
    wp = np.zeros((n_words, 5), np.float32)
    uniq, first = np.unique(wids, return_index=True)
    wp[:, 0] = 1.0
    wp[uniq, 1], wp[uniq, 2] = p[first, 1], p[first, 2]
    wp[uniq, 0] = 1.0 - wp[uniq, 1] - wp[uniq, 2]
    wp[:, 3] = np.bincount(wids, weights=loc, minlength=n_words) / np.maximum(np.bincount(wids, minlength=n_words), 1)
    np.maximum.at(wp[:, 4], wids, loc)
    return wp

MERGES = ['strict', 'lenient', 'contiguous']

def decode_words(wp, ws, text, thr=None, agg='first', merge='lenient'):
    if thr is None:
        is_loc = wp[:, 1:3].max(1) > wp[:, 0]
    else:
        is_loc = {'first': wp[:, 1] + wp[:, 2], 'mean': wp[:, 3], 'max': wp[:, 4]}[agg] > thr
    ents, cur = [], None
    for j, (l, r) in enumerate(ws):
        if not is_loc[j]:
            if cur:
                ents.append(cur)
            cur = None
            continue
        begin = wp[j, 1] >= wp[j, 2]
        if cur is None:
            if merge == 'strict' and not begin:
                continue
            cur = [l, r]
        elif begin and merge != 'contiguous':
            ents.append(cur)
            cur = [l, r]
        else:
            cur[1] = r
    if cur:
        ents.append(cur)
    return list(dict.fromkeys(text[l:r] for l, r in ents if r > l))

def decode_all(pred, texts, **params):
    return [decode_words(wp, ws, t, **params) for (wp, ws), t in zip(pred, texts)]

In [ ]:
class CharCNN(nn.Module):
    def __init__(self, n_chars, emb=32, out=64, k=3):
        super().__init__()
        self.emb = nn.Embedding(n_chars, emb, padding_idx=0)
        self.conv = nn.Conv1d(emb, out, k, padding=k // 2)
        self.out_dim = out

    def forward(self, chars):
        B, T, W = chars.shape
        c = chars.view(B * T, W)
        x = F.relu(self.conv(self.emb(c).transpose(1, 2)))
        x = x.masked_fill((c == 0).unsqueeze(1), 0.0).max(dim=2).values
        return x.view(B, T, -1)

class RNNTagger(nn.Module):
    def __init__(self, cfg, vocab_size, n_chars, n_tags, emb_weights=None):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, cfg['emb_dim'], padding_idx=PAD)
        if emb_weights is not None:
            self.word_emb.weight.data.copy_(emb_weights)
            self.word_emb.weight.requires_grad = not cfg['freeze_emb']
        self.case_emb = nn.Embedding(N_CASE, 8, padding_idx=0)
        self.char_cnn = CharCNN(n_chars) if cfg['char_cnn'] else None
        in_dim = cfg['emb_dim'] + 8 + (self.char_cnn.out_dim if self.char_cnn else 0)
        rnn_cls = {'lstm': nn.LSTM, 'gru': nn.GRU}[cfg['rnn']]
        self.rnn = rnn_cls(in_dim, cfg['hidden'], num_layers=cfg['layers'], bidirectional=cfg['bidir'],
                           batch_first=True, dropout=cfg['dropout'] if cfg['layers'] > 1 else 0.0)
        self.drop = nn.Dropout(cfg['dropout'])
        dim = cfg['hidden'] * (2 if cfg['bidir'] else 1)
        self.norm = nn.LayerNorm(dim)
        self.fc = nn.Sequential(nn.Linear(dim, dim // 2), nn.GELU(),
                                nn.Dropout(cfg['dropout']), nn.Linear(dim // 2, n_tags))
        self._init_rnn(cfg['rnn'])

    def _init_rnn(self, kind):
        n_gates = 4 if kind == 'lstm' else 3
        for name, p in self.rnn.named_parameters():
            if 'weight_hh' in name:
                for chunk in p.data.chunk(n_gates, 0):
                    nn.init.orthogonal_(chunk)
            elif 'weight_ih' in name:
                nn.init.xavier_uniform_(p.data)
            elif 'bias' in name:
                p.data.zero_()
                if kind == 'lstm' and 'bias_ih' in name:
                    H = p.shape[0] // 4
                    p.data[H:2 * H] = 1.0

    def forward(self, ids, chars, case, lengths):
        feats = [self.word_emb(ids), self.case_emb(case)] + ([self.char_cnn(chars)] if self.char_cnn else [])
        x = self.drop(torch.cat(feats, dim=-1))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))
        return self.fc(self.norm(self.drop(out)))

## 6. Checkpoints

In [ ]:
BASE = dict(
    n_uk=None, n_ru=250_000,
    emb_dim=200, char_cnn=True, freeze_emb=False,
    rnn='gru', hidden=256, layers=2, bidir=True, dropout=0.25,
    lr=1e-3, wd=0.01, epochs=5, batch_size=128,
    max_len=192, grad_clip=1.0, patience=3, max_minutes=100,
    tags='all', label_smoothing=0.0,
)
DEFAULT_DECODE = dict(thr=0.45, agg='first', merge='lenient')

In [ ]:
class NERModule(L.LightningModule):
    def __init__(self, cfg, vocab_size, n_chars, n_tags):
        super().__init__()
        self.save_hyperparameters()
        self.net = RNNTagger(cfg, vocab_size, n_chars, n_tags)
        self.gold = VAL_GOLD
        self.predictions = defaultdict(dict)

    def forward(self, batch):
        return self.net(batch['ids'], batch['chars'], batch['case'], batch['lengths'])

    def training_step(self, batch, batch_idx):
        logits = self(batch)
        loss = F.cross_entropy(logits.flatten(0, 1).float(), batch['labels'].flatten(), ignore_index=-100)
        self.log('train_loss', loss, batch_size=len(batch['idx']))
        return loss

    def validation_step(self, batch, batch_idx, dataloader_idx=0):
        key = ['uk', 'lab'][dataloader_idx]
        probs = self(batch).float().softmax(-1).cpu().numpy()
        for i, idx in enumerate(batch['idx']):
            ws = batch['word_spans'][i]
            wp = word_level_probs(probs[i, :int(batch['lengths'][i])], batch['word_ids'][i], len(ws))
            self.predictions[key][idx] = decode_words(wp, ws, batch['text'][i], **DEFAULT_DECODE)

    def on_validation_epoch_end(self):
        for key, pred in self.predictions.items():
            f1 = comp_metric(self.gold[key], [pred.get(i, []) for i in range(len(self.gold[key]))])
            self.log(f'val_{key}_f1', f1, prog_bar=(key == 'uk'))
        self.predictions.clear()

    def configure_optimizers(self):
        decay = [p for n, p in self.named_parameters() if p.requires_grad and p.ndim > 1 and 'emb' not in n]
        no_decay = [p for n, p in self.named_parameters() if p.requires_grad and (p.ndim <= 1 or 'emb' in n)]
        opt = torch.optim.AdamW([{'params': decay, 'weight_decay': self.hparams.cfg['wd']},
                                 {'params': no_decay, 'weight_decay': 0}], lr=self.hparams.cfg['lr'])
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=self.hparams.cfg['lr'],
                                                  total_steps=self.trainer.estimated_stepping_batches,
                                                  pct_start=0.1)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sch, 'interval': 'step'}}

In [ ]:
def prepare_encoder(frame, cfg):
    texts = frame.text.tolist()
    tok = WordTokenizer(min_freq=2, max_size=150_000).fit(texts)
    return Encoder(tok, TAG_SETS[cfg['tags']], texts)


def train_model(cfg):
    L.seed_everything(cfg['seed'], workers=True)
    frame = train_frame(cfg)
    enc = prepare_encoder(frame, cfg)
    train_ds = NERDataset(frame, enc, max_len=cfg['max_len'])
    val_ds = [NERDataset(VAL_SETS[k], enc) for k in ('uk', 'lab')]
    model = NERModule(cfg, enc.tok.vocab_size, len(enc.char_itos), len(enc.tags))
    directory = MODEL_DIR / cfg['name']
    directory.mkdir(parents=True, exist_ok=True)
    ckpt = ModelCheckpoint(
        dirpath=str(directory), filename='best', monitor='val_uk_f1',
        mode='max', save_top_k=1, enable_version_counter=False
    )
    trainer = L.Trainer(
        max_epochs=cfg['epochs'], max_time={'minutes': cfg['max_minutes']},
        accelerator='gpu' if GPU else 'cpu', devices=1,
        precision='16-mixed' if GPU else '32-true',
        gradient_clip_val=cfg['grad_clip'],
        logger=CSVLogger(str(OUT_DIR / 'logs'), name=cfg['name']),
        callbacks=[ckpt, EarlyStopping('val_uk_f1', patience=cfg['patience'], mode='max')],
        log_every_n_steps=50, num_sanity_val_steps=0,
    )
    trainer.fit(
        model, make_loader(train_ds, cfg['batch_size'], True),
        [make_loader(ds, 256, False) for ds in val_ds]
    )
    with open(directory / 'encoder.pkl', 'wb') as f:
        pickle.dump(enc, f)
    del model, trainer, frame, train_ds, val_ds
    gc.collect()
    if GPU:
        torch.cuda.empty_cache()


def load_model(name):
    directory = MODEL_DIR / name
    with open(directory / 'encoder.pkl', 'rb') as f:
        enc = pickle.load(f)
    ckpt = torch.load(directory / 'best.ckpt', map_location='cpu', weights_only=False)
    cfg = ckpt['hyper_parameters']['cfg']
    model = RNNTagger(cfg, enc.tok.vocab_size, len(enc.char_itos), len(enc.tags))
    model.load_state_dict({k[4:]: v for k, v in ckpt['state_dict'].items() if k.startswith('net.')})
    return model, enc

In [ ]:
EXPERIMENTS = [
    dict(name='GRU_all', rnn='gru', tags='all', seed=42),
    dict(name='LSTM_all', rnn='lstm', tags='all', seed=123),
]

for exp in EXPERIMENTS:
    path = MODEL_DIR / exp['name']
    if not all((path / f).exists() for f in ('best.ckpt', 'encoder.pkl')):
        if not TRAIN_IF_MISSING:
            raise FileNotFoundError(
                f'Missing files in {path}. Restore the trained models, '
                'or set TRAIN_IF_MISSING=True to train new ones.'
            )
        train_model({**BASE, **exp})
    print('Ready:', exp['name'])

## 7. 40-word inference

In [ ]:
ABBREV = {'м', 'вул', 'просп', 'пр', 'обл', 'р', 'с', 'смт', 'пл', 'бульв', 'пров', 'ст', 'ім', 'т', 'див', 'г', 'ул'}
SENT_END = re.compile(r'(?<=[.!?…])\s+|\s*\n\s*')

def split_chunks(text, max_chars=250):

    cuts, start = [], 0
    for m in SENT_END.finditer(text):
        prev = re.search(r'(\w+)\.$', text[start:m.start()])
        if prev and prev.group(1).lower() in ABBREV:
            continue
        cuts.append((start, m.start()))
        start = m.end()
    cuts.append((start, len(text)))
    out = []
    for a, b in cuts:
        while b - a > max_chars and (s := text.rfind(' ', a + 1, a + max_chars)) > a:
            out.append((a, s))
            a = s + 1
        out.append((a, b))
    return [(a, b) for a, b in out if WORD_RE.search(text[a:b])]

def predict_chunked(net, enc, texts, max_chars=250):

    chunks = [(i, a, b) for i, t in enumerate(texts) for a, b in split_chunks(t, max_chars)]
    chunk_pred = predict(net, NERDataset(pd.DataFrame({'text': [texts[i][a:b] for i, a, b in chunks]}), enc, labelled=False))
    wps, wss = [[] for _ in texts], [[] for _ in texts]
    for (i, a, _), (wp, ws) in zip(chunks, chunk_pred):
        wps[i].append(wp)
        wss[i].extend((l + a, r + a) for l, r in ws)
    out = []
    for i, t in enumerate(texts):
        if not wss[i]:
            out.append((np.array([[1, 0, 0, 0, 0]], np.float32), [(0, 0)]))
            continue
        assert wss[i] == word_spans(t), 'chunking changed the word grid'
        out.append((np.concatenate(wps[i]), wss[i]))
    return out

In [ ]:
@torch.no_grad()
def predict(net, ds, batch_size=256):

    net = net.to(DEVICE).eval()
    out = [None] * len(ds)
    for b in make_loader(ds, batch_size, shuffle=False):
        with torch.autocast(DEVICE, enabled=GPU):
            logits = net(b['ids'].to(DEVICE), b['chars'].to(DEVICE), b['case'].to(DEVICE), b['lengths'])
        probs = logits.float().softmax(-1).cpu().numpy()
        for i, k in enumerate(b['idx']):
            ws = b['word_spans'][i]
            out[k] = (word_level_probs(probs[i, :int(b['lengths'][i])], b['word_ids'][i], len(ws)), ws)
    return out

In [ ]:
def merge_chunks(text, max_words=40):
    parts = []
    for a, b in split_chunks(text):
        tokens = word_spans(text[a:b])
        for j in range(0, len(tokens), max_words):
            group = tokens[j:j + max_words]
            parts.append((a + group[0][0], a + group[-1][1]))

    merged = []
    start = end = None
    for a, b in parts:
        if start is None:
            start, end = a, b
        elif len(word_spans(text[start:b])) <= max_words:
            end = b
        else:
            merged.append((start, end))
            start, end = a, b
    if start is not None:
        merged.append((start, end))

    reconstructed = [
        (a + x, a + y)
        for a, b in merged
        for x, y in word_spans(text[a:b])
    ]
    assert reconstructed == word_spans(text), 'Token alignment changed'
    return merged


@torch.no_grad()
def predict_merged(net, enc, texts, max_words=40):
    chunks = [
        (i, a, b)
        for i, text in enumerate(texts)
        for a, b in merge_chunks(text, max_words)
    ]
    if not chunks:
        return [(np.array([[1, 0, 0, 0, 0]], dtype=np.float32), [(0, 0)]) for _ in texts]

    frame = pd.DataFrame({'text': [texts[i][a:b] for i, a, b in chunks]})
    chunk_pred = predict(net, NERDataset(frame, enc, labelled=False), batch_size=128)
    probs = [[] for _ in texts]
    offsets = [[] for _ in texts]
    for (i, a, _), (wp, ws) in zip(chunks, chunk_pred):
        probs[i].append(wp)
        offsets[i].extend((x + a, y + a) for x, y in ws)

    result = []
    for i, text in enumerate(texts):
        if not offsets[i]:
            result.append((np.array([[1, 0, 0, 0, 0]], dtype=np.float32), [(0, 0)]))
            continue
        wp = np.concatenate(probs[i], axis=0)
        ws = offsets[i]
        assert ws == word_spans(text), f'Token alignment failed on text {i}'
        assert len(wp) == len(ws) and np.isfinite(wp).all()
        result.append((wp, ws))
    return result

## 8. Fixed ensemble and postprocessing

In [ ]:
MODEL_NAMES = ('GRU_all', 'LSTM_all')
MODEL_WEIGHTS = (0.25, 0.75)
DECODE_PARAMS = dict(thr=0.60, agg='first', merge='strict')

FOOTER_PATTERNS = [
    r'^\s*Життя\s*🫶\s*Оболонь\s*$',
    r'^\s*Київ\s*[—–-]\s*пряма трансляція\b.*$',
    r'^\s*Хуйовий Київ\s*\|\s*Надіслати новину.*$',
]
FOOTER_RE = [re.compile(pattern, re.IGNORECASE) for pattern in FOOTER_PATTERNS]


def footer_ranges(text):
    return [
        match.span()
        for match in re.finditer(r'[^\r\n]+', text)
        if any(pattern.match(match.group()) for pattern in FOOTER_RE)
    ]


def find_occurrences(text, entity):
    if not entity:
        return []
    pattern = r'(?<!\w)' + re.escape(entity) + r'(?!\w)'
    return list(re.finditer(pattern, text, re.IGNORECASE))


def is_footer_entity(text, entity):
    matches = find_occurrences(text, entity)
    if not matches:
        return False
    ranges = footer_ranges(text)
    return all(
        any(a <= m.start() and m.end() <= b for a, b in ranges)
        for m in matches
    )


def correct_boundary(text, entity):
    entity = entity.strip()
    if '\n' in entity:
        entity = entity.splitlines()[0].strip()

    matches = find_occurrences(text, entity)
    if len(matches) == 1:
        match = matches[0]
        prefix = text[max(0, match.start() - 25):match.start()]
        street = re.search(r'(проспект|проспекті|проспекта)\s+$', prefix, re.IGNORECASE)
        if street:
            entity = street.group(1) + ' ' + entity
    return entity


def postprocess(text, locations, footer=True, boundary=True):
    output = []
    for location in locations:
        entity = correct_boundary(text, location) if boundary else location
        if footer and is_footer_entity(text, entity):
            continue
        if hard_processing(entity) not in {hard_processing(x) for x in output}:
            output.append(entity)
    return output


def predict_ensemble(texts):
    results = []
    for name in MODEL_NAMES:
        net, enc = load_model(name)
        net = net.to(DEVICE).eval()
        model_pred = []
        for start in range(0, len(texts), 256):
            model_pred.extend(predict_merged(net, enc, texts[start:start+256], max_words=40))
        results.append(model_pred)
        del net, enc
        gc.collect()
        if GPU:
            torch.cuda.empty_cache()

    ensemble = []
    for (g, ws1), (l, ws2) in zip(*results):
        assert ws1 == ws2, 'GRU/LSTM word spans do not match'
        ensemble.append((MODEL_WEIGHTS[0] * g + MODEL_WEIGHTS[1] * l, ws1))
    return ensemble


def decode_submission(texts, ensemble):
    raw = decode_all(ensemble, texts, **DECODE_PARAMS)
    final = [postprocess(text, locs) for text, locs in zip(texts, raw)]
    return raw, final

## 9. Validation (optional large-set audit)

In [ ]:
lab_probs = predict_ensemble(EVAL_TEXTS['lab'])
lab_raw, lab_final = decode_submission(EVAL_TEXTS['lab'], lab_probs)
print('Labeled F1 before:', comp_metric(VAL_GOLD['lab'], lab_raw))
print('Labeled F1 after: ', comp_metric(VAL_GOLD['lab'], lab_final))

if RUN_UK_AUDIT:
    uk_probs = predict_ensemble(EVAL_TEXTS['uk'])
    uk_raw_predictions, uk_final_predictions = decode_submission(EVAL_TEXTS['uk'], uk_probs)
    print('Ukrainian F1 before:', comp_metric(VAL_GOLD['uk'], uk_raw_predictions))
    print('Ukrainian F1 after: ', comp_metric(VAL_GOLD['uk'], uk_final_predictions))
    print('Changed Ukrainian predictions:', sum(a != b for a, b in zip(uk_raw_predictions, uk_final_predictions)))

## 10. Submission

In [ ]:
test_probs = predict_ensemble(EVAL_TEXTS['test'])
raw_locations, final_locations = decode_submission(EVAL_TEXTS['test'], test_probs)

submission = pd.DataFrame({
    'text_id': TEST.text_id.to_numpy(),
    'locations': [str(locations) for locations in final_locations],
})
assert len(submission) == len(TEST)
assert not submission.text_id.isna().any()

submission_path = OUT_DIR / 'submission_postprocessed.csv'
submission.to_csv(submission_path, index=False)
print('Changed documents:', sum(a != b for a, b in zip(raw_locations, final_locations)))
print('Saved:', submission_path)
display(submission.head())

from google.colab import files
files.download(str(submission_path))

## 11. Back up the trained models

Run this cell **after** training or inference if you need to preserve model files outside the temporary Colab runtime. It copies `best.ckpt` and `encoder.pkl` to Google Drive. The model files are required for future inference without retraining.

In [ ]:
BACKUP_TO_DRIVE = False
if BACKUP_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    backup_root = Path('/content/drive/MyDrive/rnn_competition_output/models')
    for name in MODEL_NAMES:
        target = backup_root / name
        target.mkdir(parents=True, exist_ok=True)
        for filename in ('best.ckpt', 'encoder.pkl'):
            shutil.copy2(MODEL_DIR / name / filename, target / filename)
    print('Models saved to:', backup_root)